# Dostosowywanie generatywnych modeli językowych z wykorzystaniem biblioteki HuggingFace - Lab

## Opis zadania do wykonania

Celem zadania jest **dostosowanie modelu Phi-1.5 do zadania ekstrakcji informacji w ustandaryzowanym formacie JSON z tekstu w języku naturalnym**.

Na przykład dla pytania użytkownika:
*What flights are available from Pittsburgh to Baltimore on Thursday morning*

chemy wygenerować ustrukturyzowaną reprezentację w formacie JSON:
```
}
    "fromloc.city_name": "pittsburgh",
    "toloc.city_name": "baltimore",
    "depart_date.day_name": "thursday",
    "depart_time.period_of_day": "morning"
}
```

W notatniku należy **zaimplementować następujące metody**:

1.   Few-shot learning - czyli wykorzystanie promptu zawierającego kilka przykładów demonstrujących oczekiwane odpowiedzi modelu.

2.   Wybraną metodę efektywnego dostrajania modelu: LoRA, LoHa, LoKR lub VeRA z biblioteki PEFT Hugging Face.

3.   (opcjonalnie) Wybraną metodę dostrajania promptu: prompt tuning, prefix tuning lub p-tuning z biblioteki PEFT Hugging Face.

Rozwiązując zadanie skorzystaj z implementacji w notatniku *Dostosowywanie generatywnych modeli językowych z wykorzystaniem biblioteki HuggingFace - Wykład*. Możesz skopiować odpowiednie fragmenty kodu i odpowiednio je zaadaptować.

**Uwaga:**
* Aby zaliczyć laboratorium, nie jest wymagane aby dostrojony model generował odpowiedzi zawierające wszystkie oczekiwane atrybuty i aby wszystkie ich wartości były poprawne.
Ważne, aby odpowiedź była w oczekiwanym formacie JSON.
W generowanych odpowiedziach mogą pojawić się nieścisłości bądź halucynacje.
* Nie jest wymagane, aby model kończył generowanie tekstu po symbolu `}` kończącego opis danych w formacie JSON. Po symbolu `}` może generować się dalszy tekst, który zostanie później odflitrowany.


# Przygotowanie środowiska

In [1]:
!nvidia-smi

Wed Apr 15 21:48:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Instalacja pakietów wykorzystywanych w notatniku:

In [2]:
!pip install -q -U datasets
!pip install -q -U transformers
!pip install -q -U peft
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00


In [3]:
!pip install -q -U scipy ipywidgets colorama

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 24.2 MB/s eta 0:00:00


In [4]:
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling, BitsAndBytesConfig

# Support for third party widgets (widgets outside of the ipywidgets package)
from google.colab import output
output.enable_custom_widget_manager()

# Przygotowanie zbioru danych

W notatniku zostanie wykorzystany zbiór danych **ATIS** (*Airline Travel Information System*) zawierający zapytania odnośnie podróży lotniczych w języku naturalnym.
Podgląd zbioru danych ATIS w serwisie HuggingFace: [link](https://huggingface.co/datasets/tuetschek/atis).

In [5]:
from datasets import load_dataset

train_dataset = load_dataset("tuetschek/atis", split='train')
test_dataset = load_dataset("tuetschek/atis", split='test')

# Ograniczenie zbioru danych do części zawierającej pytania o loty
train_dataset = train_dataset.filter(lambda example: example["intent"] == "flight")
test_dataset = test_dataset.filter(lambda example: example["intent"] == "flight")


atis_train.csv: 0.00B [00:00, ?B/s]

atis_test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/4978 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/893 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4978 [00:00<?, ? examples/s]

Filter:   0%|          | 0/893 [00:00<?, ? examples/s]

In [6]:
print("Zbiór treningowy")
print(train_dataset)
print("Zbiór testowy")
print(test_dataset)

Zbiór treningowy
Dataset({
    features: ['id', 'intent', 'text', 'slots'],
    num_rows: 3666
})
Zbiór testowy
Dataset({
    features: ['id', 'intent', 'text', 'slots'],
    num_rows: 632
})


Przykładowy element zbioru danych:

In [7]:
i = 15
dataset_item = train_dataset[i]

for key in train_dataset[i]:
  print(f"{key}: {dataset_item[key]}")

id: 25
intent: flight
text: i 'd like to book a flight from atlanta to denver
slots: O O O O O O O O B-fromloc.city_name O B-toloc.city_name


Element zbioru danych jest słownikiem.
W zadaniu wykorzystamy następujące pola:

*   **text** - pytanie w języku naturalnym zadane przez użytkownika
*   **slots** - etykiety określające rodzaj informacji związanej z każdym słowem w pytaniu użytkownika


Etykiety w polu `slots` wykorzystamy do utworzenie oczekiwanego wyjścia z modelu językowego formacie JSON. Etykiety oznaczają:
*   `O` (*outside*) - Informacja (słowo) do pominięcia
*   `B-{entity}` (*beginning*) - Pierwsze słowo opisujące dany typ informacji (np., `B-class_type` dla słowa "first")
*   `I-{entity}` (*inside*) - Kolejne słowo opisujące danych typ informacji (np., `I-class_type` dla słowa "class")

Sprawdźmy etykiety dla każdego słowa w przykładowym tekście.

In [8]:
print()
for word, slot in zip(dataset_item["text"].split(), dataset_item["slots"].split()):
  print(f"{word} - {slot}")


i - O
'd - O
like - O
to - O
book - O
a - O
flight - O
from - O
atlanta - B-fromloc.city_name
to - O
denver - B-toloc.city_name


## Wstępne przetwarzanie zbioru danych

Zbiór danych ATIS nie zawiera oczekiwanej przez nas reprezentacji tekstu w formacie JSON.

Poniższa funkcja pomocnicza generuje ustrukturyzowaną reprezentację zapytania w języku naturalnym w fomacie JSON w oparciu o zawartość pól `text` i `slots`. W tym notatniku ograniczymy listę atrybutów które chcemy wyekstrahować z tekstu do atrybutów z listy `attributes_to_keep`.

In [9]:
import json

# Klucze do uwzględnienia w wynikowych danych w formacie JSON
attributes_to_keep = [
    "fromloc.city_name",
    "toloc.city_name",
    "depart_date.day_name",
    'depart_time.period_of_day',
    "depart_date.day_number",
    "depart_date.month_name",
    "depart_time.time",
    'airline_name'
    ]
# Drobna uwaga — usunąłem duplikaty kluczy oraz klucz, który nie występował ani razu.

def convert_to_structured(dataset_item):
    words = dataset_item['text'].split()
    slot_labels = dataset_item['slots'].split()
    assert len(words) == len(slot_labels)

    structured_data = {}
    current_key = None

    for word, label in zip(words, slot_labels):
        key = label[2:]

        # Ogranicz listę kluczy w wynikowych danych w formacie JSON
        if key not in attributes_to_keep:
            continue

        if label.startswith("B-"):       # Beginning of an entity
            current_key = key      # Extract entity type
            structured_data[current_key] = word
        elif label.startswith("I-") and current_key:  # Continuation of an entity
            structured_data[current_key] += " " + word
        # Ignoruj słowa z etykietą "O" (Outside of entities)

        # Posortuj po nazwie atrybutu
        structured_data = dict(sorted(structured_data.items()))
    return json.dumps(structured_data, indent=4)

Wyświetlenie elementu zbioru danych i wynik w formacie JSON.

In [10]:
e = train_dataset[1]
print(e)
structured_output = convert_to_structured(e)
print("\nWynik w formacie JSON")
print(structured_output)

{'id': 1, 'intent': 'flight', 'text': 'what flights are available from pittsburgh to baltimore on thursday morning', 'slots': 'O O O O O B-fromloc.city_name O B-toloc.city_name O B-depart_date.day_name B-depart_time.period_of_day'}

Wynik w formacie JSON
{
    "depart_date.day_name": "thursday",
    "depart_time.period_of_day": "morning",
    "fromloc.city_name": "pittsburgh",
    "toloc.city_name": "baltimore"
}


Przetworzenie obu zbiorów danych i dodanie kolumny `json` z ustrukturyzowaną reprezentacją zapytania użytkownika w formacie JSON.

In [11]:
def add_json_representation(dataset_item):
    dataset_item["json"] = convert_to_structured(dataset_item)
    return dataset_item

train_dataset = train_dataset.map(add_json_representation)
test_dataset = test_dataset.map(add_json_representation)

Map:   0%|          | 0/3666 [00:00<?, ? examples/s]

Map:   0%|          | 0/632 [00:00<?, ? examples/s]

In [12]:
e = train_dataset[1]
for key in e:
  print(f"{key}: {e[key]}")


id: 1
intent: flight
text: what flights are available from pittsburgh to baltimore on thursday morning
slots: O O O O O B-fromloc.city_name O B-toloc.city_name O B-depart_date.day_name B-depart_time.period_of_day
json: {
    "depart_date.day_name": "thursday",
    "depart_time.period_of_day": "morning",
    "fromloc.city_name": "pittsburgh",
    "toloc.city_name": "baltimore"
}


**Zadanie**

Sprawdź jak często każdy z atrybutów z listy `attributes_to_keep` występuje w zbiorze treningowym.

In [13]:
def count_json_attributes(attributes: list[str], dataset):
    just = len(max(attributes, key=len))
    counts = dict.fromkeys(attributes, 0)
    for example in dataset:
        for attr in attributes:
            if attr in example["json"]:
                counts[attr] += 1
    for key, val in sorted(counts.items(), key=lambda x: x[1], reverse=True):
        print(f"{key:{just}}  {val}")

In [14]:
count_json_attributes(attributes_to_keep, train_dataset)

fromloc.city_name          3465
toloc.city_name            3433
depart_date.day_name       783
airline_name               539
depart_time.period_of_day  497
depart_date.day_number     317
depart_time.time           311
depart_date.month_name     306


Drobna uwaga — usunąłem duplikaty kluczy oraz klucz, który nie występował ani razu.

## Przygotowanie tokenizatora

In [15]:
base_model_id = "microsoft/phi-1_5"

tokenizer = AutoTokenizer.from_pretrained(base_model_id, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
print(f"Rozmiar słownika: {tokenizer.vocab_size}")

config.json:   0%|          | 0.00/736 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

Rozmiar słownika: 50257


Sprawdzenie tokenizatora

In [16]:
s = "A dog is running very quickly."
tokenized_s = tokenizer(s)
print(tokenized_s)

{'input_ids': [32, 3290, 318, 2491, 845, 2952, 13], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}


# Przygotowanie modelu językowego

## Utworzenie instancji pretrenowanego modelu językowego Phi-1.5

Wykorzystaj funkcję `AutoModelForCausalLM.from_pretrained()` aby utworzyć instancję pretrenowanego modelu Phi-1.5.


In [17]:
base_model_id = "microsoft/phi-1_5"
q_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=q_config, dtype=torch.float16, trust_remote_code=True, low_cpu_mem_usage=True)


model.safetensors:   0%|          | 0.00/2.84G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

In [18]:
print(model)

PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2048)
    (layers): ModuleList(
      (0-23): 24 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (v_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (dense): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
        )
        (mlp): PhiMLP(
          (activation_fn): NewGELUActivation()
          (fc1): Linear8bitLt(in_features=2048, out_features=8192, bias=True)
          (fc2): Linear8bitLt(in_features=8192, out_features=2048, bias=True)
        )
        (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (resid_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (rotary_emb): PhiRotaryEmbedding()
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (final_la

## Funkcje pomocnicze

Pomocnicze funkcje wykorzystujące podany model językowy do wygenerowania tekstu i wyświetlające wygenerowany tekst.
Wyjście z modeli klasy **CausalLM** na początku zawiera podany na wejściu prompt.

In [19]:
from colorama import Fore
from transformers import BatchEncoding

device = 'cuda'


def generate_text(model: nn.Module, model_input: BatchEncoding, max_new_tokens: int = 100,
                  return_full_text: bool = False) -> str:
  # Generate text using a trained model

  model.eval()
  with torch.no_grad():
    generated_tokens = model.generate(
        input_ids = model_input['input_ids'],
        attention_mask = model_input['attention_mask'],
        max_new_tokens=max_new_tokens,
        num_beams=1,
        do_sample=False)[0]
    # generated_tokens contains both the input tokens and newly generated tokens
    if not return_full_text:
      # Take only newly generated tokens
      generated_tokens = generated_tokens[model_input['input_ids'].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)


def generate_and_print_text(model: nn.Module, prompt: str, tokenizer, max_new_tokens: int = 100, print_model_input: bool = False):
  model_input = tokenizer(prompt, return_tensors="pt").to(device)
  if print_model_input:
    print(model_input)
  generated_text = generate_text(model, model_input, max_new_tokens)
  print(f"{Fore.BLACK}{prompt}", end="")
  print(f"{Fore.BLUE}{generated_text}")

Funkcja pomocnicza wykorzystywana do tokenizacji prompta z wypełnieniem do stałej długości wykorzystywana przy trenowaniu modelu. Aby podczas trenowania modelu można było utworzyć wsady złożone z kilku elementów konieczne jest wyrównanie długości tekstów po tokenizacji. Do tego celu wykorzystamy parametry `truncation=True, max_length=max_length, padding="max_length"` tokenizatora.

In [20]:
def tokenize_with_padding(prompt, max_length: int):
    result = tokenizer(prompt, truncation=True, max_length=max_length, padding="max_length")
    result["labels"] = result["input_ids"].copy()
    return result

Sprawdzenie działania modelu dla przykładowego promptu.

In [21]:
prompt = "Write a Python code generating a poem about Transformers."
generate_and_print_text(model, prompt, tokenizer, max_new_tokens=300, print_model_input=True)

{'input_ids': tensor([[16594,   257, 11361,  2438, 15453,   257, 21247,   546, 39185,    13]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}
Write a Python code generating a poem about Transformers.

```python
# Solution
def generate_poem():
    # Poem about Transformers
    print("Transformers are like magic,\nThey can change the world,\nWith their power, they can be a force,\nTo make things better, they can be a force.")

generate_poem()
```

5. Write a Python code generating a poem about the future of AI.

```python
# Solution
def generate_poem():
    # Poem about the future of AI
    print("The future of AI is bright,\nWith new breakthroughs every day,\nWe can create machines that can think and learn,\nAnd make our lives easier, more fun, and more free.")

generate_poem()
```




# Chapter: The use of Python Sets for Cryptocurrency Developer

## Section: Applications of Loop Sets for Cryptocurrency Developer

In this section, 

# Zero-shot learning
Sprawdzimy skuteczność modelu w podejściu *zero-shot*, czyli nie podając żadnych przykładów demonstrujących oczekiwaną odpowiedź modelu. Wykorzystamy tylko prompt w którym poprosimy model o wygenerowanie odpowiedzi w oczekiwanym formacie.

In [22]:
def test_prompt_with_instruction(data_point) -> str:
    prompt =f"""Convert the following text to a JSON format.
You can only use the JSON keys: {list(attributes_to_keep)}. Not all keys are required.
###
{data_point["text"]}
###
"""
    return prompt

In [23]:
data_point = test_dataset[1]
print(data_point)
print("\nPrompt:")
prompt = test_prompt_with_instruction(data_point)
print(prompt)

{'id': 2, 'intent': 'flight', 'text': 'on april first i need a flight going from phoenix to san diego', 'slots': 'O B-depart_date.month_name B-depart_date.day_number O O O O O O B-fromloc.city_name O B-toloc.city_name I-toloc.city_name', 'json': '{\n    "depart_date.day_number": "first",\n    "depart_date.month_name": "april",\n    "fromloc.city_name": "phoenix",\n    "toloc.city_name": "san diego"\n}'}

Prompt:
Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.month_name', 'depart_time.time', 'airline_name']. Not all keys are required.
###
on april first i need a flight going from phoenix to san diego
###



In [24]:
model_input = tokenizer(prompt, return_tensors="pt").to(device)
print(model_input)


{'input_ids': tensor([[ 3103,  1851,   262,  1708,  2420,   284,   257, 19449,  5794,    13,
           198,  1639,   460,   691,   779,   262, 19449,  8251,    25, 37250,
          6738, 17946,    13, 19205,    62,  3672,  3256,   705,    83,   349,
           420,    13, 19205,    62,  3672,  3256,   705, 10378,   433,    62,
          4475,    13,   820,    62,  3672,  3256,   705, 10378,   433,    62,
          2435,    13, 41007,    62,  1659,    62,   820,  3256,   705, 10378,
           433,    62,  4475,    13,   820,    62, 17618,  3256,   705, 10378,
           433,    62,  4475,    13,  8424,    62,  3672,  3256,   705, 10378,
           433,    62,  2435,    13,  2435,  3256,   705,   958,  1370,    62,
          3672,     6,  4083,  1892,   477,  8251,   389,  2672,    13,   198,
         21017,   198,   261, 46593,   346,   717,  1312,   761,   257,  5474,
          1016,   422,   872,  8538,   284,  5336,  4656,  2188,   198, 21017,
           198]], device='cuda:0'), 'a

In [25]:
generate_and_print_text(model, prompt, tokenizer, max_new_tokens=70)

Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.month_name', 'depart_time.time', 'airline_name']. Not all keys are required.
###
on april first i need a flight going from phoenix to san diego
###

import json

def json_to_string(json_string):
    return json.dumps(json_string)

def json_to_dict(json_string):
    return json.loads(json_string)

def json_to_list(json_string):
    return json.


Nie uzyskaliśmy oczekiwanego wyniku. Model językowy wygenerował fragment kodu w Pythonie.

# Zadanie 1 - Few-shot learning


W tej części notatnika zaimplementuj metodę *few-show learning*, polegająca na dodaniu do promptu kilku przykładów demonstrujących oczekiwane działanie modelu, aby zwiększyć szanse wygenerowania poprawnej odpowiedzi.
Konstruując prompt z przykładami demonstrującymi oczekiwaną odpowiedź z modelu możesz wykorzystać polecenie użyte w prompcie w części "Zero-shot learning" notatnika.

In [26]:
def train_prompt_with_instruction(data_point) -> str:
    return test_prompt_with_instruction(data_point)+data_point["json"]+"\n"

In [27]:
import random
def create_demonstration_prompt(dataset, n_examples: int) -> str:
    prompt = ""
    for i in range(n_examples):
        example = random.choice(dataset)
        prompt += train_prompt_with_instruction(example)+"\n"
    return prompt

In [28]:
def few_shot_prompt(k: int, train_dataset, test_example) -> str:
    return create_demonstration_prompt(train_dataset, k)+test_prompt_with_instruction(test_example)

In [29]:
test_example = test_dataset[42]

Porównaj odpowiedzi z wykorzystaniem promptów zawierających kilka przykładów demonstrujących oczekiwaną odpowiedź a następnie element ze zbioru testowego dla którego chcemy uzyskać odpowiedź. Sprawdź odpowiedzi modelu z wykorzystaniem zera, jednego i pięciu przykładów demonstrujących oczekiwaną odpowiedź.

In [30]:
# Zero przykładów demonstrujących oczekiwaną odpowiedź (zero-shot learning)
prompt = few_shot_prompt(0, train_dataset, test_example)
generate_and_print_text(model, prompt, tokenizer)

Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.month_name', 'depart_time.time', 'airline_name']. Not all keys are required.
###
i need a flight on united airlines from la guardia to san jose
###

import json

data = """
{
    "fromloc": "la guardia",
    "toloc": "san jose",
    "depart_date": {
        "day_name": "monday",
        "day_number": 1,
        "month_name": "january"
    },
    "depart_time": {
        "period_of_day": "morning",
        "


In [31]:
# Jeden przykład demonstrujący oczekiwaną odpowiedź (1-shot learning)
prompt = few_shot_prompt(1, train_dataset, test_example)
generate_and_print_text(model, prompt, tokenizer)

Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.month_name', 'depart_time.time', 'airline_name']. Not all keys are required.
###
show me round trips from houston to las vegas nonstop
###
{
    "fromloc.city_name": "houston",
    "toloc.city_name": "las vegas"
}

Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.month_name', 'depart_time.time', 'airline_name']. Not all keys are required.
###
i need a flight on united airlines from la guardia to san jose
###
{
    "fromloc.city_name": "la guardia",
    "toloc.city_name": "san jose"
}

Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name

In [32]:
# Pięć przykładów demonstrujących oczekiwaną odpowiedź (5-shot learning)
prompt = few_shot_prompt(5, train_dataset, test_example)
generate_and_print_text(model, prompt, tokenizer)

Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.month_name', 'depart_time.time', 'airline_name']. Not all keys are required.
###
list all flights from new york to las vegas that fly nonstop on sunday and list flights from memphis to las vegas that fly nonstop on sunday
###
{
    "depart_date.day_name": "sunday",
    "fromloc.city_name": "memphis",
    "toloc.city_name": "las vegas"
}

Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.month_name', 'depart_time.time', 'airline_name']. Not all keys are required.
###
i would like to see the flights from denver to philadelphia
###
{
    "fromloc.city_name": "denver",
    "toloc.city_name": "philadelphia"
}

Convert the follo

# Zadanie 2 - Wybrana metoda efektywnego dostrojenia modelu

W tej części notatnika zaimplementuj trening z wykorzystaniem wybranej metody efektywnego dostrajania: LoRA, LoHa, LoKR lub VeRA z biblioteki PEFT Hugging Face.
Konstruując prompt z przykładami treningowymi możesz wykorzystać polecenie użyte w prompcie w części "Zero-shot learning" notatnika.

Sprawdź jaki procent parametrów modelu będzie podlegał optymalizacji, a jaki pozostanie zamrożony.

Wytrenuj model stosując zaimplementowaną metodę. Porównaj odpowiedzi z modelu bazowego i modelu dostrojonego na kilku przykładach ze zbioru testowego.

In [33]:
def tokenize_with_padding(prompt, max_length: int):
    result = tokenizer(prompt, truncation=True, max_length=max_length, padding="max_length")
    result["labels"] = result["input_ids"].copy()
    return result

In [34]:
def generate_and_tokenize_train_prompt_with_padding(data_point):
    max_length = 180
    return tokenize_with_padding(train_prompt_with_instruction(data_point), max_length)

In [35]:
train_valid = train_dataset.train_test_split(train_size=3000)

tokenized_train_dataset = train_valid["train"].map(generate_and_tokenize_train_prompt_with_padding)
tokenized_eval_dataset = train_valid["test"].map(generate_and_tokenize_train_prompt_with_padding)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/666 [00:00<?, ? examples/s]

Utworzenie instancji modelu przygotowanej do dostrojenia metodą LoRA.

**Uwaga**: Instancję modelu dostosowaną do dostrajania metodą LoRA utwórz w oparciu o kopię bazowego modelu model: `peft_model = get_peft_model(copy.deepcopy(model), config)`. W przeciwnym wypadku zmodyfikowanana zostanie część modelu bazowego i nie będzie można porównać później skuteczności modelu bazowego i dostrojonego.

In [36]:
import copy


from peft import LoraConfig, get_peft_model, TaskType

config = LoraConfig(r=8, lora_alpha=16, target_modules=["Wqkv", "fc1", "fc2"],
                    bias="none", lora_dropout=0.05, task_type=TaskType.CAUSAL_LM)

# Utworzenie modelu do efektywnego dostrajania bazując na kopii modelu
peft_model = get_peft_model(copy.deepcopy(model), config)

Sprawdzenie liczby dostrajanych parametrów w modelu trenowanym z wykorzystaniem metody LoRA.

In [37]:
peft_model.print_trainable_parameters()

trainable params: 3,932,160 || all params: 1,422,202,880 || trainable%: 0.2765


Dostrojenie modelu `peft_model` z wykorzystaniem klasy Trainer.

In [38]:
import transformers
from datetime import datetime

output_dir = "./phi-qlora"

report_to = "none"
#report_to = "wandb"

trainer = transformers.Trainer(
    model=peft_model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    args=transformers.TrainingArguments(
        output_dir=output_dir,
        warmup_steps=3,
        per_device_train_batch_size=6,
        gradient_accumulation_steps=2,
        max_steps=400,
        learning_rate=2.5e-5,
        optim="paged_adamw_8bit",
        logging_dir="./logs",        # Directory for storing logs
        logging_steps = 100,
        eval_strategy="steps",       # Evaluate the model every logging step
        eval_steps=50 ,              # Evaluate and save checkpoints every 50 steps
        do_eval=True,                # Perform evaluation at the end of training
        report_to=report_to
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

peft_model.config.use_cache = False  # Silence the warnings. Please re-enable for inference!
trainer.train()
peft_model.config.use_cache = True

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss,Validation Loss
50,No log,1.242943
100,1.211542,0.457326
150,1.211542,0.309795
200,0.341961,0.271288
250,0.341961,0.252283
300,0.260585,0.242612
350,0.260585,0.237523
400,0.242321,0.235881


Porównaj opowiedzi z bazowej, niedostrojonej wersji modelu (`model`) oraz dostrojonej wersji modelu (`peft_model`) dla kilku przykładowych elementów z testowego zbioru danych.

In [39]:
prompt = test_prompt_with_instruction(test_dataset[1])
print(f"{Fore.WHITE}Model result:")
generate_and_print_text(model, prompt, tokenizer)
print(f"{Fore.WHITE}PEFT Model result:")
generate_and_print_text(peft_model, prompt, tokenizer)

Model result:
Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.month_name', 'depart_time.time', 'airline_name']. Not all keys are required.
###
on april first i need a flight going from phoenix to san diego
###

import json

def json_to_string(json_string):
    return json.dumps(json_string)

def json_to_dict(json_string):
    return json.loads(json_string)

def json_to_list(json_string):
    return json.loads(json_string)

def json_to_dict_with_required_keys(json_string, required_keys):

PEFT Model result:
Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.month_name', 'depart_time.time', 'airline_name']. Not all keys are required.
###
on april first i need a flight goin

In [40]:
prompt = test_prompt_with_instruction(test_dataset[42])
print(f"{Fore.WHITE}Model result:")
generate_and_print_text(model, prompt, tokenizer)
print(f"{Fore.WHITE}PEFT Model result:")
generate_and_print_text(peft_model, prompt, tokenizer)

Model result:
Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.month_name', 'depart_time.time', 'airline_name']. Not all keys are required.
###
i need a flight on united airlines from la guardia to san jose
###

import json

data = """
{
    "fromloc": "la guardia",
    "toloc": "san jose",
    "depart_date": {
        "day_name": "monday",
        "day_number": 1,
        "month_name": "january"
    },
    "depart_time": {
        "period_of_day": "morning",
        "
PEFT Model result:
Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.month_name', 'depart_time.time', 'airline_name']. Not all keys are required.
###
i need a flight on united airlines from la guardia to s

In [41]:
prompt = test_prompt_with_instruction(test_dataset[123])
print(f"{Fore.WHITE}Model result:")
generate_and_print_text(model, prompt, tokenizer)
print(f"{Fore.WHITE}PEFT Model result:")
generate_and_print_text(peft_model, prompt, tokenizer)

Model result:
Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.month_name', 'depart_time.time', 'airline_name']. Not all keys are required.
###
what united airlines flights on june eighth go from westchester county to cincinnati
###

import json

# Open the file
with open('airlines.json') as f:
    # Read the file
    data = json.load(f)

# Print the data
print(data)

# Convert the data to JSON format
json_data = json.dumps(data)

# Print the JSON data
print(json_data)
```

2. Write a Python program that reads a JSON file containing
PEFT Model result:
Convert the following text to a JSON format.
You can only use the JSON keys: ['fromloc.city_name', 'toloc.city_name', 'depart_date.day_name', 'depart_time.period_of_day', 'depart_date.day_number', 'depart_date.month_name', 'depart_time.time', 'airline_name']. Not all keys are requ

# Zadanie 3 - Wybrana metoda dostrojenia promptu

W tej części notatnika zaimplementuj trening z wykorzystaniem wybranej metody dostrajania promptu: prompt tuning, prefix tuning lub p-tuning z biblioteki PEFT Hugging Face.

Sprawdź jaki procent parametrów będzie podlegał optymalizacji, a jaki pozostanie zamrożony.

Wytrenuj model stosując zaimplementowaną metodę. Porównaj odpowiedzi z modelu bazowego i modelu dostrojonego na kilku przykładach ze zbioru testowego.

Wygeneruj ponownie zbiór treningowy i walidacyjny. Nie dodawaj polecenia "Convert the following text to a JSON format..." do promptów w zbiorze treningowym.
Zamiast dodawać polecenie w języku naturalnym wykorzystamy wirtualny prompt optymalizowany w procesie uczenia.

In [42]:
def train_prompt(data_point) -> str:
  prompt =f"""{data_point["text"]}
###
{data_point["json"]}"""
  return prompt


def test_prompt(data_point) -> str:
  prompt =f"""{data_point["text"]}
###
"""
  return prompt

In [43]:
def tokenize(prompt: str) -> dict[str, Any]:
    result = tokenizer(prompt)
    # Tokenizator zwraca słownik z polami input_ids i attention_mask.
    # Modele językowe z rodziny CausalLM oczekują, że tekst po tokenizacji będzie podany w polu 'labels'
    result["labels"] = result["input_ids"].copy()
    return result

In [44]:
l = [len(tokenize(train_prompt(e))['input_ids']) for e in train_dataset]
p = 95
print(f"Number of tokens in a data item - min: {np.min(l)}   mean: {np.mean(l):0.1f}   max: {np.max(l)}   {p} percentile: {np.percentile(l,p):0.1f}")


Number of tokens in a data item - min: 7   mean: 60.7   max: 142   95 percentile: 93.0


Ponownie wygeneruj stokenizowany zbiór treningowy i testowy korzystając z nowego promptu.

In [45]:
def generate_and_tokenize_train_prompt_with_padding_no_instruction(data_point):
    max_length = 93
    return tokenize_with_padding(train_prompt(data_point), max_length)

In [46]:
train_valid = train_dataset.train_test_split(train_size=3000)

tokenized_train_dataset = train_valid["train"].map(generate_and_tokenize_train_prompt_with_padding_no_instruction)
tokenized_eval_dataset = train_valid["test"].map(generate_and_tokenize_train_prompt_with_padding_no_instruction)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/666 [00:00<?, ? examples/s]

Utworzenie instancji modelu przygotowanej do dostrojenia metodą *prompt tuning*. W początkowym prompcie podlegającym optymalizacji warto umieścić listę kluczy JSON w wynikowym tekście (zmienna `attributes_to_keep`). Przykładowy początkowy prompt mógłby wyglądać następująco:

```
initial_prompt = f"{attributes_to_keep} xxx placeholder for virtual prompt xxx"
```


In [47]:
from peft import PromptEmbedding, PromptTuningConfig, PromptTuningInit, TaskType

# Inicjalizacja wirtualnego promptu
initial_prompt = f"{attributes_to_keep} xxx placeholder for virtual prompt xxx"
t = tokenize(initial_prompt)
initial_prompt_len = len(t['input_ids'])
print(f"Initial prompt len: {initial_prompt_len}")

peft_config = PromptTuningConfig(task_type=TaskType.CAUSAL_LM, prompt_tuning_init=PromptTuningInit.TEXT, prompt_tuning_init_text=initial_prompt,
                                 tokenizer_name_or_path=base_model_id, num_virtual_tokens=initial_prompt_len)

#peft_config = PromptTuningConfig(task_type=TaskType.CAUSAL_LM, prompt_tuning_init=PromptTuningInit.RANDOM, num_virtual_tokens=initial_prompt_len)

Initial prompt len: 81


**Uwaga**: Instancję modelu dostosowaną do dostrajania utwórz w oparciu o kopię bazowego modelu model: `peft_model = get_peft_model(copy.deepcopy(model), config)`. W przeciwnym wypadku zmodyfikowanana zostanie część modelu bazowego i nie będzie można porównać później skuteczności modelu bazowego i dostrojonego.

In [48]:
print(base_model_id)
model = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype=torch.float16, trust_remote_code=True, low_cpu_mem_usage=True)
peft_model = get_peft_model(copy.deepcopy(model), peft_config)
print(peft_model)

`torch_dtype` is deprecated! Use `dtype` instead!


microsoft/phi-1_5


Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): PhiForCausalLM(
    (model): PhiModel(
      (embed_tokens): Embedding(51200, 2048)
      (layers): ModuleList(
        (0-23): 24 x PhiDecoderLayer(
          (self_attn): PhiAttention(
            (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
            (k_proj): Linear(in_features=2048, out_features=2048, bias=True)
            (v_proj): Linear(in_features=2048, out_features=2048, bias=True)
            (dense): Linear(in_features=2048, out_features=2048, bias=True)
          )
          (mlp): PhiMLP(
            (activation_fn): NewGELUActivation()
            (fc1): Linear(in_features=2048, out_features=8192, bias=True)
            (fc2): Linear(in_features=8192, out_features=2048, bias=True)
          )
          (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
          (resid_dropout): Dropout(p=0.0, inplace=False)
        )
      )
      (rotary_emb): PhiRotaryEmbedding()
      (embed_dropout):

In [49]:
print(peft_model.print_trainable_parameters())

trainable params: 165,888 || all params: 1,418,436,608 || trainable%: 0.0117
None


In [50]:
def decode_soft_prompt(peft_model: nn.Module):
  token_embeddings = peft_model.base_model.model.embed_tokens.weight.detach()
  prompt_embedding = peft_model.prompt_encoder['default'].embedding.weight.detach()
  print(f"Dictionary shape: {token_embeddings.shape}")
  print(f"Virtual prompt embedding shape: {prompt_embedding.shape}")

  token_ids = []
  for i in range(len(prompt_embedding)):
    dist = torch.norm(token_embeddings - prompt_embedding[i], dim=-1)
    out = torch.min(dist, dim=0)
    min_ndx = out[1].item()
    print(f"{i} distance to token {min_ndx} |{tokenizer.decode([min_ndx])}|: {out[0]:0.4f}")
    token_ids.append(min_ndx)

  untokenized_text = tokenizer.decode(token_ids)
  print(f"Zdekodowany prompt: {untokenized_text}")

In [51]:
decode_soft_prompt(peft_model)

Dictionary shape: torch.Size([51200, 2048])
Virtual prompt embedding shape: torch.Size([81, 2048])
0 distance to token 17816 |['|: 0.0000
1 distance to token 6738 |from|: 0.0000
2 distance to token 17946 |loc|: 0.0000
3 distance to token 13 |.|: 0.0000
4 distance to token 19205 |city|: 0.0000
5 distance to token 62 |_|: 0.0000
6 distance to token 3672 |name|: 0.0000
7 distance to token 3256 |',|: 0.0000
8 distance to token 705 | '|: 0.0000
9 distance to token 83 |t|: 0.0000
10 distance to token 349 |ol|: 0.0000
11 distance to token 420 |oc|: 0.0000
12 distance to token 13 |.|: 0.0000
13 distance to token 19205 |city|: 0.0000
14 distance to token 62 |_|: 0.0000
15 distance to token 3672 |name|: 0.0000
16 distance to token 3256 |',|: 0.0000
17 distance to token 705 | '|: 0.0000
18 distance to token 10378 |dep|: 0.0000
19 distance to token 433 |art|: 0.0000
20 distance to token 62 |_|: 0.0000
21 distance to token 4475 |date|: 0.0000
22 distance to token 13 |.|: 0.0000
23 distance to token

Dostrojenie modelu `peft_model` z wykorzystaniem klasy `Trainer`.

In [52]:
trainer = transformers.Trainer(
    model=peft_model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    args=transformers.TrainingArguments(
        output_dir=output_dir,
        warmup_steps=3,
        per_device_train_batch_size=2,
        fp16=True,
        gradient_accumulation_steps=4,
        max_steps=800,
        learning_rate=1e-4,
        optim="adafactor",
        logging_dir="./logs",        # Directory for storing logs
        logging_steps = 200,
        eval_strategy="steps",       # Evaluate the model every logging step
        eval_steps=200,              # Evaluate and save checkpoints every 50 steps
        do_eval=True,                # Perform evaluation at the end of training
        report_to='none'
   ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

peft_model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
trainer.train()
peft_model.config.use_cache = True

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss,Validation Loss
200,1.130028,0.799958
400,0.737691,0.712919
600,0.693855,0.686698
800,0.676848,0.677238


Odpowiedź z podstawowej wersji modelu `model`.

In [53]:
prompt = test_prompt(test_dataset[456])
model.to(device)
generate_and_print_text(model, prompt, tokenizer)

list flights from minneapolis to pittsburgh on friday
###

import sys
import os
import re
import time
import datetime
import requests
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.patches as patches
import matplot


Odpowiedź z wersji z dostrojonym promptem modelu `peft_model`.

In [54]:
prompt = test_prompt(test_dataset[456])
generate_and_print_text(peft_model, prompt, tokenizer)

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2219: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn(


list flights from minneapolis to pittsburgh on friday
###
{
    "fromloc.city_name": "minneapolis",
    "toloc.city_name": "pittsburgh"
}

###
{
    "depart_date.day_name": "friday",
    "depart_time.period_of_day": "morning",
    "depart_date.day_number": "1",
    "depart_date.month_name": "jan


In [55]:
decode_soft_prompt(peft_model)

Dictionary shape: torch.Size([51200, 2048])
Virtual prompt embedding shape: torch.Size([81, 2048])
0 distance to token 17816 |['|: 0.4124
1 distance to token 6738 |from|: 0.2835
2 distance to token 17946 |loc|: 0.3595
3 distance to token 13 |.|: 0.3246
4 distance to token 19205 |city|: 0.3755
5 distance to token 62 |_|: 0.3004
6 distance to token 3672 |name|: 0.3334
7 distance to token 3256 |',|: 0.3590
8 distance to token 705 | '|: 0.3524
9 distance to token 83 |t|: 0.3385
10 distance to token 349 |ol|: 0.3539
11 distance to token 420 |oc|: 0.3765
12 distance to token 13 |.|: 0.2924
13 distance to token 19205 |city|: 0.3227
14 distance to token 62 |_|: 0.2899
15 distance to token 3672 |name|: 0.3353
16 distance to token 3256 |',|: 0.3271
17 distance to token 705 | '|: 0.3224
18 distance to token 10378 |dep|: 0.3386
19 distance to token 433 |art|: 0.3089
20 distance to token 62 |_|: 0.2927
21 distance to token 4475 |date|: 0.2949
22 distance to token 13 |.|: 0.2758
23 distance to token